# Portfolio Evaluation: CNN-GARCH vs Baselines

Ce notebook implémente l'**évaluation financière** du modèle CNN-GARCH hybride.  
Au lieu de mesurer la précision statistique (log-MSE), on mesure la **qualité de l'allocation de portefeuille**.

## Pipeline
```
Returns (N stocks) → Modèle → σ_i (annualisée ×252) → W_i = (1/σ²_i)/Σ(1/σ²_j)
                                                           ↓
                                            r_portfolio = Σ(W_i × r_i)
                                                           ↓
                                            σ_portfolio, Sharpe, Drawdown
```

## Stratégies comparées
| Stratégie | Description |
|-----------|-------------|
| **NN-GARCH** | σ prédite par le CNN hybride (notre modèle) |
| **Hist-Vol** | Volatilité historique rolling (fenêtre glissante) |
| **EWMA** | Volatilité exponentielle pondérée (RiskMetrics λ=0.94) |
| **ARCH(1,1)** | Modèle ARCH classique fitté sur le train |
| **Equal-Weight** | 1/N benchmark |

**Référence superviseur** :  
- Annualisation : ×252  
- W = (1/σ²) / Σ(1/σ²)  
- **Loss actuelle (entraînement)** : MSE(log σ², log σ²_GARCH)  
- **Loss cible (superviseur)** : MSE(σ_pred, |r|)  
- r_portfolio = Σ(r_it × W_i)

## 1. Imports & Configuration

In [ ]:
from __future__ import annotations

import warnings
import sys
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from model import HybridGarch
from data_stocks_WIKI_price import get_cleaned_data

print(f"PyTorch {torch.__version__}")
print(f"NumPy   {np.__version__}")
print(f"Pandas  {pd.__version__}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
FILE_PATH   = r"C:/Users/karim/OneDrive/Documents/Msc AI/Finance/TP/WIKI_PRICES_212b326a081eacca455e13140d7bb9db.zip"
CHECKPOINT  = "finetune_real_zeroaware_r2min1e8_best.pt"   # meilleur checkpoint
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WINDOW_SIZE = 90      # fenêtre d'entrée du CNN (même que l'entraînement)
N_STOCKS    = 30      # nombre d'actions dans le portefeuille
TRAIN_RATIO = 0.70    # 70% train / 30% test (out-of-sample)
ANNUALIZE   = 252     # jours de trading par an

EWMA_LAMBDA = 0.94    # paramètre RiskMetrics EWMA
REFIT_ARCH  = False   # True = ARCH re-fitté (très lent), False = fit unique sur train

print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CHECKPOINT}")
print(f"Window     : {WINDOW_SIZE} | Stocks : {N_STOCKS} | Train : {int(TRAIN_RATIO*100)}%")

## 2. Chargement des données

In [ ]:
print("Loading WIKI Prices data...")
returns_df_full = get_cleaned_data(FILE_PATH)
print(f"Full universe : {returns_df_full.shape[0]} days × {returns_df_full.shape[1]} stocks")
print(f"Date range    : {returns_df_full.index[0].date()} → {returns_df_full.index[-1].date()}")

# Sélection des N_STOCKS actions (déjà filtrées pour être complètes)
selected = returns_df_full.columns[:N_STOCKS].tolist()
returns_df = returns_df_full[selected].copy()
print(f"\nPortefeuille : {N_STOCKS} actions")
print(f"Tickers      : {selected[:10]} ...")

In [ ]:
# ── Train / Test split ───────────────────────────────────────────────────────
n_total   = len(returns_df)
split_idx = int(TRAIN_RATIO * n_total)

returns_train = returns_df.iloc[:split_idx]
returns_test  = returns_df.iloc[split_idx:]

# Statistiques par action calculées sur le train uniquement
train_means = returns_train.mean().values.astype(np.float32)  # [N_STOCKS]
train_stds  = returns_train.std(ddof=0).values.astype(np.float32)  # [N_STOCKS]

print(f"Train : {returns_train.index[0].date()} → {returns_train.index[-1].date()} ({len(returns_train)} jours)")
print(f"Test  : {returns_test.index[0].date()} → {returns_test.index[-1].date()} ({len(returns_test)} jours)")

## 3. Chargement du modèle CNN-GARCH

In [ ]:
model = HybridGarch(hidden=32, kernel=3, conv_layers=5)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modèle chargé : {CHECKPOINT}")
print(f"Paramètres    : {n_params:,}")
print(f"Device        : {DEVICE}")

## 4. Fonctions d'estimation de volatilité

Pour chaque méthode, on estime $\sigma^2_i(t)$ **annualisée** $= \sigma^2_{daily} \times 252$.

In [ ]:
@torch.no_grad()
def _predict_nn_batch(windows_norm: np.ndarray, model: torch.nn.Module,
                      device: torch.device) -> np.ndarray:
    """Forward pass batché : [N_stocks, W] → σ²_norm [N_stocks]."""
    x = torch.tensor(windows_norm, dtype=torch.float32, device=device).unsqueeze(-1)  # [N, W, 1]
    sigma2_norm, _, _, _ = model(x)
    return sigma2_norm.cpu().numpy()


def precompute_sigma2_nn(returns_arr: np.ndarray,
                         train_means: np.ndarray, train_stds: np.ndarray,
                         model: torch.nn.Module, device: torch.device,
                         window_size: int, start_t: int,
                         annualize: int = 252) -> np.ndarray:
    """
    Pour chaque t ∈ [start_t, T), prédit σ² annualisée pour toutes les actions
    en une seule passe batché (toutes actions simultanément).
    Retourne [T - start_t, N_stocks].
    """
    T, N = returns_arr.shape
    test_len = T - start_t
    sigma2_mat = np.full((test_len, N), np.nan)

    eps = 1e-8
    for k, t in enumerate(range(start_t, T)):
        windows = returns_arr[t - window_size:t, :]  # [W, N]
        if np.isnan(windows).any():
            continue
        # Normalisation par stock (train stats)
        windows_T = windows.T  # [N, W]
        windows_norm = (windows_T - train_means[:, None]) / (train_stds[:, None] + eps)

        s2_norm = _predict_nn_batch(windows_norm, model, device)  # [N]
        s2_real = s2_norm * (train_stds + eps) ** 2               # rescale
        sigma2_mat[k] = np.clip(s2_real, 1e-10, None) * annualize

    return sigma2_mat


def precompute_sigma2_hist(returns_arr: np.ndarray,
                           window_size: int, start_t: int,
                           annualize: int = 252) -> np.ndarray:
    """Volatilité historique rolling. Retourne [test_len, N_stocks]."""
    T, N = returns_arr.shape
    test_len = T - start_t
    sigma2_mat = np.zeros((test_len, N))

    for k, t in enumerate(range(start_t, T)):
        window = returns_arr[t - window_size:t, :]
        std_vals = np.nanstd(window, axis=0, ddof=0)
        sigma2_mat[k] = std_vals ** 2 * annualize

    return sigma2_mat


def precompute_sigma2_ewma(returns_arr: np.ndarray,
                           lam: float, start_t: int,
                           annualize: int = 252) -> np.ndarray:
    """
    EWMA RiskMetrics : σ²[t] = λ·σ²[t-1] + (1-λ)·r²[t-1].
    Retourne [test_len, N_stocks].
    """
    T, N = returns_arr.shape
    test_len = T - start_t
    sigma2_mat = np.zeros((test_len, N))

    # Initialisation : variance sur les 30 premiers jours
    s2 = np.var(returns_arr[:30, :], axis=0, ddof=0)
    s2 = np.maximum(s2, 1e-10)

    for t in range(1, T):
        s2 = lam * s2 + (1 - lam) * returns_arr[t - 1, :] ** 2
        if t >= start_t:
            k = t - start_t
            sigma2_mat[k] = s2 * annualize

    return sigma2_mat


def precompute_sigma2_arch(returns_arr: np.ndarray,
                           train_means: np.ndarray,
                           split_idx: int, start_t: int,
                           annualize: int = 252) -> np.ndarray:
    """
    ARCH(1,1) fitté une fois sur le train, puis récursion forward pour le test.
    Retourne [test_len, N_stocks].
    """
    from arch import arch_model as arch_fit

    T, N = returns_arr.shape
    test_len = T - start_t
    sigma2_mat = np.zeros((test_len, N))

    r_train = returns_arr[:split_idx, :]

    for j in range(N):
        r_j = r_train[:, j].astype(np.float64)
        try:
            am  = arch_fit(r_j * 100, mean='Zero', vol='GARCH', p=1, q=1, dist='normal')
            res = am.fit(disp='off')
            omega_j = res.params['omega'] / (100 ** 2)
            alpha_j = res.params['alpha[1]']
            beta_j  = res.params['beta[1]']
            init_s2 = float(res.conditional_volatility[-1] ** 2)
        except Exception:
            # Fallback si fit échoue
            omega_j, alpha_j, beta_j = 1e-6, 0.05, 0.90
            init_s2 = np.var(r_j, ddof=0)

        # Récursion forward sur [split_idx, T)
        s2 = init_s2
        for t in range(split_idx, T):
            r_prev = returns_arr[t - 1, j]
            s2 = omega_j + alpha_j * r_prev ** 2 + beta_j * s2
            s2 = max(s2, 1e-10)
            if t >= start_t:
                sigma2_mat[t - start_t, j] = s2 * annualize

    return sigma2_mat


print("Fonctions d'estimation définies.")

## 5. Précalcul des matrices de volatilité

Pour chaque méthode et chaque jour du **test set**, on calcule $\sigma^2_i(t)$ annualisée pour toutes les actions.

In [ ]:
returns_arr = returns_df.values.astype(np.float32)  # [T, N]
start_t = max(WINDOW_SIZE, split_idx)                # premier jour où on peut prédire
test_dates = returns_df.index[start_t:]

print(f"Précalcul sur {len(test_dates)} jours de test, {N_STOCKS} actions...")
print(f"(start_t = {start_t}, correspondant à {returns_df.index[start_t].date()})\n")

# ── NN-GARCH ─────────────────────────────────────────────────────────────────
print("[1/4] NN-GARCH (forward pass batché)...")
t0 = time.time()
sigma2_nn = precompute_sigma2_nn(
    returns_arr, train_means, train_stds,
    model, DEVICE, WINDOW_SIZE, start_t, ANNUALIZE
)
print(f"      OK — {time.time()-t0:.1f}s")

# ── Hist-Vol ──────────────────────────────────────────────────────────────────
print("[2/4] Historical Volatility...")
t0 = time.time()
sigma2_hist = precompute_sigma2_hist(returns_arr, WINDOW_SIZE, start_t, ANNUALIZE)
print(f"      OK — {time.time()-t0:.1f}s")

# ── EWMA ─────────────────────────────────────────────────────────────────────
print("[3/4] EWMA (RiskMetrics λ=0.94)...")
t0 = time.time()
sigma2_ewma = precompute_sigma2_ewma(returns_arr, EWMA_LAMBDA, start_t, ANNUALIZE)
print(f"      OK — {time.time()-t0:.1f}s")

# ── ARCH(1,1) ────────────────────────────────────────────────────────────────
print("[4/4] ARCH(1,1) (fit sur train + récursion)...")
t0 = time.time()
sigma2_arch = precompute_sigma2_arch(
    returns_arr.astype(np.float64), train_means, split_idx, start_t, ANNUALIZE
)
print(f"      OK — {time.time()-t0:.1f}s")

print("\nToutes les matrices σ² calculées.")
print(f"Shape de chaque matrice : {sigma2_nn.shape}  [test_days × N_stocks]")

## 6. Construction du portefeuille

Poids d'allocation inverse de la variance :
$$W_i(t) = \frac{1/\sigma^2_i(t)}{\sum_j 1/\sigma^2_j(t)}$$

Return du portefeuille :
$$r_{\text{portfolio}}(t) = \sum_i W_i(t) \cdot r_i(t)$$

In [ ]:
def inverse_variance_weights(sigma2: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """W_i = (1/σ²_i) / Σ(1/σ²_j). Gère les NaN en les remplaçant par la moyenne."""
    s2 = sigma2.copy()
    nan_mask = np.isnan(s2) | (s2 <= 0)
    if nan_mask.all():
        return np.ones(len(s2)) / len(s2)
    # Remplacer NaN par la moyenne des valeurs valides
    s2[nan_mask] = np.nanmean(s2[~nan_mask])
    inv_var = 1.0 / np.maximum(s2, eps)
    return inv_var / inv_var.sum()


def compute_portfolio_returns(sigma2_mat: np.ndarray,
                               returns_arr: np.ndarray,
                               start_t: int,
                               strategy: str = 'inv-var') -> tuple[np.ndarray, np.ndarray]:
    """
    Calcule les returns journaliers du portefeuille et les poids.
    Returns:
        port_returns : [test_len]  — returns journaliers
        weights_hist : [test_len, N_stocks] — poids à chaque jour
    """
    test_len = sigma2_mat.shape[0]
    N = sigma2_mat.shape[1]
    port_returns = np.zeros(test_len)
    weights_hist = np.zeros((test_len, N))

    for k in range(test_len):
        t = start_t + k
        r_t = returns_arr[t]  # returns réalisés au jour t

        if strategy == 'inv-var':
            w = inverse_variance_weights(sigma2_mat[k])
        else:  # equal-weight
            w = np.ones(N) / N

        port_returns[k] = np.dot(w, r_t)
        weights_hist[k] = w

    return port_returns, weights_hist


# ── Calcul des returns pour chaque stratégie ────────────────────────────────
test_len = len(test_dates)

print("Construction des portefeuilles...")

r_nn,   w_nn   = compute_portfolio_returns(sigma2_nn,   returns_arr, start_t)
r_hist, w_hist = compute_portfolio_returns(sigma2_hist, returns_arr, start_t)
r_ewma, w_ewma = compute_portfolio_returns(sigma2_ewma, returns_arr, start_t)
r_arch, w_arch = compute_portfolio_returns(sigma2_arch, returns_arr, start_t)
r_eq,   w_eq   = compute_portfolio_returns(sigma2_nn,   returns_arr, start_t, strategy='equal')

# DataFrame des returns journaliers
port_returns = pd.DataFrame({
    'NN-GARCH'     : r_nn,
    'Hist-Vol'     : r_hist,
    'EWMA'         : r_ewma,
    'ARCH(1,1)'    : r_arch,
    'Equal-Weight' : r_eq,
}, index=test_dates)

print(f"Returns calculés — {len(port_returns)} jours de test.")
print("\nMoyenne des returns journaliers (%) :")
print((port_returns.mean() * 100).round(4).to_string())

## 7. Métriques financières

In [ ]:
def financial_metrics(returns: pd.Series, annualize: int = 252) -> dict:
    """Calcule les métriques clés d'un portefeuille."""
    r = returns.dropna()

    ann_ret = r.mean() * annualize
    ann_vol = r.std(ddof=0) * np.sqrt(annualize)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-10 else np.nan

    cumret      = (1 + r).cumprod()
    rolling_max = cumret.cummax()
    drawdown    = (cumret - rolling_max) / rolling_max
    max_dd      = float(drawdown.min())

    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 1e-10 else np.nan

    # Turnover moyen (approximatif via vol journalière des returns)
    daily_vol_pct = ann_vol / np.sqrt(annualize) * 100

    return {
        'Ann. Return (%)'   : round(ann_ret * 100, 3),
        'Ann. Volatility (%)': round(ann_vol * 100, 3),
        'Sharpe Ratio'      : round(sharpe, 4),
        'Max Drawdown (%)'  : round(max_dd * 100, 3),
        'Calmar Ratio'      : round(calmar, 4),
        'Daily Vol (%)'     : round(daily_vol_pct, 4),
    }


metrics = pd.DataFrame(
    {col: financial_metrics(port_returns[col], ANNUALIZE) for col in port_returns.columns}
).T

print("=" * 70)
print("PERFORMANCE DU PORTEFEUILLE — OUT-OF-SAMPLE")
print("=" * 70)
print(metrics.to_string())
print("=" * 70)

# Rang par volatilité (critère principal : minimiser σ_portfolio)
print("\nClassement par volatilité annualisée (objectif : minimiser) :")
rank = metrics['Ann. Volatility (%)'].sort_values()
for i, (name, val) in enumerate(rank.items(), 1):
    marker = " ← MEILLEUR" if i == 1 else ""
    print(f"  {i}. {name:<18} {val:.3f}%{marker}")

## 8. Visualisations

In [ ]:
COLORS = {
    'NN-GARCH'     : '#e63946',
    'Hist-Vol'     : '#457b9d',
    'EWMA'         : '#2a9d8f',
    'ARCH(1,1)'    : '#e9c46a',
    'Equal-Weight' : '#adb5bd',
}
STRATEGIES = list(COLORS.keys())

fig = plt.figure(figsize=(18, 16))
gs  = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.32)

# ── 1. Returns cumulatifs ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
for s in STRATEGIES:
    cumret = (1 + port_returns[s]).cumprod()
    ax1.plot(cumret.index, cumret.values, label=s,
             color=COLORS[s], linewidth=1.8 if s == 'NN-GARCH' else 1.2,
             zorder=3 if s == 'NN-GARCH' else 2)
ax1.axhline(1, color='black', linewidth=0.5, linestyle='--')
ax1.set_title('Returns Cumulatifs (Out-of-Sample)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Valeur du Portefeuille (base 1)')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── 2. Volatilité rolling (63 jours ≈ 3 mois) ──────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
roll_w = 63
for s in STRATEGIES:
    rv = port_returns[s].rolling(roll_w).std() * np.sqrt(ANNUALIZE) * 100
    ax2.plot(rv.index, rv.values, label=s, color=COLORS[s],
             linewidth=1.8 if s == 'NN-GARCH' else 1.0)
ax2.set_title(f'Volatilité Annualisée Rolling ({roll_w}j) (%)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Vol. Annualisée (%)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# ── 3. Drawdowns ────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
for s in STRATEGIES:
    cumret = (1 + port_returns[s]).cumprod()
    dd = (cumret - cumret.cummax()) / cumret.cummax() * 100
    ax3.fill_between(dd.index, dd.values, 0, alpha=0.35,
                     color=COLORS[s], label=s)
    ax3.plot(dd.index, dd.values, color=COLORS[s], linewidth=0.7)
ax3.set_title('Drawdown (%)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Drawdown (%)')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

# ── 4. Bar chart : volatilité annualisée ────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
vols = metrics['Ann. Volatility (%)'].sort_values()
bars = ax4.bar(vols.index, vols.values,
               color=[COLORS[k] for k in vols.index],
               edgecolor='black', linewidth=0.6)
ax4.set_title('Volatilité Annualisée par Stratégie (%)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Vol. Annualisée (%)')
ax4.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, vols.values):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f'{val:.2f}%', ha='center', va='bottom', fontsize=9)
ax4.grid(axis='y', alpha=0.3)

# ── 5. Bar chart : Sharpe Ratio ─────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
sharpes = metrics['Sharpe Ratio']
bars = ax5.bar(sharpes.index, sharpes.values,
               color=[COLORS[k] for k in sharpes.index],
               edgecolor='black', linewidth=0.6)
ax5.axhline(0, color='black', linewidth=0.8)
ax5.set_title('Sharpe Ratio par Stratégie', fontsize=11, fontweight='bold')
ax5.set_ylabel('Sharpe Ratio')
ax5.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, sharpes.values):
    offset = 0.01 if val >= 0 else -0.05
    ax5.text(bar.get_x() + bar.get_width() / 2, val + offset,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax5.grid(axis='y', alpha=0.3)

plt.suptitle(
    f'Évaluation Portefeuille — CNN-GARCH vs Baselines\n'
    f'({N_STOCKS} actions, test : {test_dates[0].year}–{test_dates[-1].year})',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('portfolio_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : portfolio_comparison.png")

## 9. Analyse des poids NN-GARCH

In [ ]:
w_nn_df   = pd.DataFrame(w_nn,   index=test_dates, columns=returns_df.columns)
w_hist_df = pd.DataFrame(w_hist, index=test_dates, columns=returns_df.columns)
w_ewma_df = pd.DataFrame(w_ewma, index=test_dates, columns=returns_df.columns)

fig, axes = plt.subplots(3, 1, figsize=(16, 13))

# ── Heatmap poids NN ────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(w_nn_df.T.values, aspect='auto', cmap='YlOrRd',
               extent=[0, len(w_nn_df), -0.5, N_STOCKS - 0.5],
               vmin=0, vmax=w_nn_df.max().max())
ax.set_title('Poids NN-GARCH au cours du temps (Heatmap)', fontsize=11, fontweight='bold')
ax.set_xlabel('Jours de test')
ax.set_ylabel('Actions')
ax.set_yticks(range(N_STOCKS))
ax.set_yticklabels(returns_df.columns, fontsize=6)
plt.colorbar(im, ax=ax, label='Poids')

# ── Top 5 stocks : évolution des poids ────────────────────────────────────
ax2 = axes[1]
mean_weights_nn = w_nn_df.mean().sort_values(ascending=False)
top5 = mean_weights_nn.head(5).index.tolist()
for stock in top5:
    smoothed = w_nn_df[stock].rolling(21).mean()
    ax2.plot(w_nn_df.index, smoothed, label=f"{stock}", linewidth=1.5)
ax2.set_title('Top 5 actions — Poids NN-GARCH (MA 21j)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Poids dans le portefeuille')
ax2.legend()
ax2.grid(alpha=0.3)

# ── Comparaison : concentration des poids (Herfindahl index) ───────────────
ax3 = axes[2]
hhi_nn   = (w_nn_df   ** 2).sum(axis=1)   # Σ w²_i → concentration
hhi_hist = (w_hist_df ** 2).sum(axis=1)
hhi_ewma = (w_ewma_df ** 2).sum(axis=1)
hhi_eq   = 1.0 / N_STOCKS  # constant

ax3.plot(hhi_nn.rolling(21).mean(),   label='NN-GARCH',  color=COLORS['NN-GARCH'],  linewidth=1.5)
ax3.plot(hhi_hist.rolling(21).mean(), label='Hist-Vol',  color=COLORS['Hist-Vol'],  linewidth=1.2)
ax3.plot(hhi_ewma.rolling(21).mean(), label='EWMA',      color=COLORS['EWMA'],      linewidth=1.2)
ax3.axhline(hhi_eq, color=COLORS['Equal-Weight'], linestyle='--', label='Equal-Weight', linewidth=1)
ax3.set_title('Indice de Concentration des Poids (Herfindahl — MA 21j)\n'
              'Plus élevé = plus concentré | 1/N = diversification max',
              fontsize=11, fontweight='bold')
ax3.set_ylabel('Σ w²_i')
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('weights_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : weights_analysis.png")

## 10. Diagnostic : σ_pred vs |r| (point 3 du superviseur)

Le superviseur demande une loss `MSE(σ_pred, |r|)`.  
Cette cellule montre la **corrélation actuelle** entre σ prédite et |r| réalisé,  
ce qui motive le changement de loss pour le prochain entraînement.

In [ ]:
# Diagnostic sur les N_STOCKS actions — σ_pred vs |r_t|
print("Calcul σ_pred vs |r| pour diagnostics...")

sigma_pred_all = np.sqrt(np.maximum(sigma2_nn / ANNUALIZE, 1e-10))  # σ daily [test_len, N]
abs_r_all      = np.abs(returns_arr[start_t:])                       # |r| daily [test_len, N]

# Corrélation moyenne σ vs |r| par action
corrs = []
for j in range(N_STOCKS):
    mask = ~(np.isnan(sigma_pred_all[:, j]) | np.isnan(abs_r_all[:, j]))
    if mask.sum() > 10:
        corr = np.corrcoef(sigma_pred_all[mask, j], abs_r_all[mask, j])[0, 1]
        corrs.append(corr)
corr_mean = np.nanmean(corrs)

print(f"Corrélation moyenne σ_pred vs |r|  : {corr_mean:.4f} (sur {N_STOCKS} actions)")

# ── Plot sur une action exemple ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

stock0 = returns_df.columns[0]
j0 = 0
sp0 = sigma_pred_all[:, j0]
ar0 = abs_r_all[:, j0]
mask0 = ~(np.isnan(sp0) | np.isnan(ar0))

# Scatter
ax = axes[0]
ax.scatter(ar0[mask0], sp0[mask0], alpha=0.15, s=4, color='#e63946')
m = max(ar0[mask0].max(), sp0[mask0].max())
ax.plot([0, m], [0, m], 'k--', linewidth=1, label='y = x (parfait)')
c = np.corrcoef(ar0[mask0], sp0[mask0])[0, 1]
ax.set_title(f'{stock0} — σ_pred vs |r|  (corr={c:.3f})', fontsize=11, fontweight='bold')
ax.set_xlabel('|r_t| (réalisé)')
ax.set_ylabel('σ_pred (NN-GARCH)')
ax.legend()
ax.grid(alpha=0.3)

# Time series (premiers 500 jours)
ax2 = axes[1]
N_show = min(500, len(test_dates))
ax2.plot(test_dates[:N_show], ar0[:N_show], label='|r_t|', color='gray', alpha=0.7, linewidth=0.7)
ax2.plot(test_dates[:N_show], sp0[:N_show], label='σ_pred (NN)', color='#e63946', linewidth=1.3)
ax2.set_title(f'{stock0} — σ_pred vs |r_t| (premiers {N_show}j)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Volatilité journalière')
ax2.legend()
ax2.grid(alpha=0.3)

# Distribution des corrélations par action
ax3 = axes[2]
ax3.hist(corrs, bins=15, color='#457b9d', edgecolor='black', linewidth=0.5)
ax3.axvline(corr_mean, color='red', linestyle='--', linewidth=1.5,
            label=f'Moyenne = {corr_mean:.3f}')
ax3.set_title('Distribution des corrélations σ_pred vs |r|\n(sur toutes les actions)',
              fontsize=11, fontweight='bold')
ax3.set_xlabel('Corrélation')
ax3.set_ylabel('Nombre d\'actions')
ax3.legend()
ax3.grid(alpha=0.3)

plt.suptitle('Diagnostic : σ_pred (NN-GARCH) vs |r| réalisé — Motivation du changement de loss',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('sigma_vs_absr_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : sigma_vs_absr_diagnostic.png")

## 11. Résumé final

In [ ]:
# ── Tableau de synthèse ──────────────────────────────────────────────────────
print("=" * 70)
print("RÉSUMÉ FINAL — COMPARAISON PORTEFEUILLES")
print(f"Période test : {test_dates[0].date()} → {test_dates[-1].date()}")
print(f"Univers      : {N_STOCKS} actions")
print(f"Annualisation: ×{ANNUALIZE} | Fenêtre CNN : {WINDOW_SIZE}j")
print("=" * 70)
print(metrics.to_string())
print("=" * 70)

best_vol    = metrics['Ann. Volatility (%)'].idxmin()
best_sharpe = metrics['Sharpe Ratio'].idxmax()
best_dd     = metrics['Max Drawdown (%)'].idxmax()  # le moins négatif

print(f"\nMeilleure stratégie (volatilité)  : {best_vol}")
print(f"Meilleure stratégie (Sharpe)      : {best_sharpe}")
print(f"Meilleure stratégie (Drawdown)    : {best_dd}")

# Amélioration relative NN vs Equal-Weight
vol_nn  = metrics.loc['NN-GARCH',     'Ann. Volatility (%)']
vol_eq  = metrics.loc['Equal-Weight', 'Ann. Volatility (%)']
improv  = (vol_eq - vol_nn) / vol_eq * 100
print(f"\nRéduction de volatilité NN vs Equal-Weight : {improv:+.2f}%")

# Diagnostic σ_pred
print(f"\nCorrélation σ_pred vs |r| (loss cible) : {corr_mean:.4f}")
if corr_mean > 0.15:
    print("  → Le modèle capture bien les régimes de volatilité.")
else:
    print("  → Faible corrélation : le fine-tuning avec loss MSE(σ, |r|) est recommandé.")